# Starter Notebook

Install and import required libraries

In [1]:
!pip install transformers datasets evaluate accelerate peft trl bitsandbytes
!pip install nvidia-ml-py3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 2.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia

In [2]:
import os
import pandas as pd
import torch
from transformers import RobertaModel, RobertaTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding, RobertaForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from datasets import load_dataset, Dataset, ClassLabel
import pickle
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch.nn.functional as F

2025-04-11 23:35:38.628400: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744414538.815378      73 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744414538.868187      73 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Load Tokenizer and Preprocess Data

In [3]:

import re
#文本长度过滤
def length_filter(example, tokenizer, min_len=5, max_len=256):
    tokenized = tokenizer(example["text"])
    return min_len <= len(tokenized["input_ids"]) <= max_len

#重复文本过滤
seen = set()
def is_unique(example):
    if example["text"] in seen:
        return False
    seen.add(example["text"])
    return True

#过滤含有大量特殊字符（如 @#$%^&*~）的样本
def has_too_many_symbols(text, threshold=0.3):
    symbols = re.findall(r"[^\w\s]", text)
    return len(symbols) / max(len(text), 1) < threshold


def combined_filter(example):
    return (
        length_filter(example, tokenizer, 5, 256)
        and has_too_many_symbols(example["text"])
        and is_unique(example)
    )

In [4]:
#augment
import random


soft_words = ["really", "actually", "definitely", "simply", "just", "kind of"]

def augment_text(text):
    words = text.split()
    if len(words) < 5:
        return text  

    op = random.choice(["soft_insert", "swap", "repeat"])

    if op == "soft_insert":
        idx = random.randint(0, len(words))
        words.insert(idx, random.choice(soft_words))

    elif op == "swap":
        idx = random.randint(0, len(words) - 2)
        words[idx], words[idx+1] = words[idx+1], words[idx]

    elif op == "repeat":
        idx = random.randint(0, len(words) - 1)
        words.insert(idx, words[idx]) 
    return " ".join(words)

In [5]:


base_model = 'roberta-base'

dataset = load_dataset('ag_news', split='train')
tokenizer = RobertaTokenizer.from_pretrained(base_model)
dataset = dataset.filter(combined_filter)
print("过滤后数据量：", len(dataset))

def preprocess(examples):
    text = examples["text"]
    if random.random() < 0.2:
        text = augment_text(text)
    
    tokenized = tokenizer(text, truncation=True, padding=True)
    
    return tokenized

tokenized_dataset = dataset.map(preprocess, batched=False, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

过滤后数据量： 119955


Map:   0%|          | 0/119955 [00:00<?, ? examples/s]

In [6]:
print("训练集大小：", len(tokenized_dataset))
if len(tokenized_dataset) == 0:
    raise ValueError("❌ 训练集为空，检查 filter 或 map 是否问题！")

训练集大小： 119955


In [7]:
# Extract the number of classess and their names
num_labels = dataset.features['label'].num_classes
class_names = dataset.features["label"].names
print(f"number of labels: {num_labels}")
print(f"the labels: {class_names}")

# Create an id2label mapping
# We will need this for our classifier.
id2label = {i: label for i, label in enumerate(class_names)}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")


number of labels: 4
the labels: ['World', 'Sports', 'Business', 'Sci/Tech']


## Load Pre-trained Model
Set up config for pretrained model and download it from hugging face

In [8]:




model = RobertaForSequenceClassification.from_pretrained(
    base_model,
    id2label=id2label)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Anything from here on can be modified

In [9]:
# Split the original training set
split_datasets = tokenized_dataset.train_test_split(test_size=640, seed=42)
train_dataset = split_datasets['train']
eval_dataset = split_datasets['test']
collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [10]:
#train teacher

teacher_model = RobertaForSequenceClassification.from_pretrained(
    base_model,
    id2label=id2label
)

teacher_args = TrainingArguments(
    output_dir="teacher_model",
    report_to="none",
    eval_strategy='steps',
    logging_steps=100,
    learning_rate=2e-5,
    num_train_epochs=1,
    max_steps=1500,
    use_cpu=False,
    weight_decay=0.01,
    label_smoothing_factor=0.0,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    dataloader_num_workers=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    optim="adamw_torch",
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={'use_reentrant':True},
    save_strategy="no"
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_score": f1}

teacher_trainer = Trainer(
    model=teacher_model,
    args=teacher_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    
    compute_metrics=compute_metrics
)

teacher_trainer.train()

teacher_logits = []
teacher_model.eval()
with torch.no_grad():
    for batch in torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=collator):
        inputs = {k: v.to(teacher_model.device) for k, v in batch.items() if k != "labels"}
        outputs = teacher_model(**inputs).logits
        teacher_logits.extend(outputs.detach().cpu())

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_73/3969022585.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  teacher_trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,F1 Score
100,1.196700,0.411143,0.882812,0.885151
200,0.369500,0.314015,0.896875,0.898492
300,0.336100,0.257415,0.912500,0.913467
400,0.323400,0.305906,0.895312,0.896753
500,0.317900,0.279337,0.904687,0.906076
600,0.309600,0.240662,0.918750,0.920301
700,0.283400,0.221779,0.932813,0.933866
800,0.261800,0.223289,0.925000,0.926477
900,0.289400,0.203271,0.928125,0.929198
1000,0.274800,0.201567,0.935937,0.936779


## Setup LoRA Config
Setup PEFT config and get peft model for finetuning

In [11]:
# PEFT Config
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias = 'none',
    target_modules = ['query', 'key', 'value'],
    task_type="SEQ_CLS",
)

In [12]:
peft_model = get_peft_model(model, peft_config)
peft_model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): RobertaForSequenceClassification(
      (roberta): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(50265, 768, padding_idx=1)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): Mod

In [13]:
print("Trainable parameters:")
for name, param in peft_model.named_parameters():
    if param.requires_grad:
        print(name)

Trainable parameters:
base_model.model.roberta.encoder.layer.0.attention.self.query.lora_A.default.weight
base_model.model.roberta.encoder.layer.0.attention.self.query.lora_B.default.weight
base_model.model.roberta.encoder.layer.0.attention.self.key.lora_A.default.weight
base_model.model.roberta.encoder.layer.0.attention.self.key.lora_B.default.weight
base_model.model.roberta.encoder.layer.0.attention.self.value.lora_A.default.weight
base_model.model.roberta.encoder.layer.0.attention.self.value.lora_B.default.weight
base_model.model.roberta.encoder.layer.1.attention.self.query.lora_A.default.weight
base_model.model.roberta.encoder.layer.1.attention.self.query.lora_B.default.weight
base_model.model.roberta.encoder.layer.1.attention.self.key.lora_A.default.weight
base_model.model.roberta.encoder.layer.1.attention.self.key.lora_B.default.weight
base_model.model.roberta.encoder.layer.1.attention.self.value.lora_A.default.weight
base_model.model.roberta.encoder.layer.1.attention.self.value.

In [14]:
print('PEFT Model')
peft_model.print_trainable_parameters()

PEFT Model
trainable params: 1,036,036 || all params: 125,684,744 || trainable%: 0.8243


## Training Setup

In [15]:
# To track evaluation accuracy during training


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    # Calculate accuracy
    accuracy = accuracy_score(labels, preds)
    f1=f1_score(labels,preds, average='macro')
    return {
        'accuracy': accuracy,
        'f1_score':f1
    }

In [16]:
# Distillation loss (combined with cross entropy + KL)
def distill_loss_fn(student_logits, teacher_logits, true_labels, temperature=2.0, alpha=0.5):
    ce = F.cross_entropy(student_logits, true_labels)
    kl = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=-1),
        F.softmax(teacher_logits / temperature, dim=-1),
        reduction="batchmean"
    ) * (temperature ** 2)
    return alpha * ce + (1 - alpha) * kl

In [17]:
# Override compute_loss
class DistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False,  **kwargs):
        labels = inputs["labels"]
        idx = inputs["input_ids"].cpu().numpy()
        with torch.no_grad():
            teacher_out = teacher_model(**{k: v.to(teacher_model.device) for k, v in inputs.items() if k != "labels"})
        student_out = model(**inputs)
        loss = distill_loss_fn(student_out.logits, teacher_out.logits, labels)
        return (loss, student_out) if return_outputs else loss

In [18]:
# student
student_args = TrainingArguments(
    output_dir="student_model",
    report_to="none",
    eval_strategy='steps',
    logging_steps=100,
    learning_rate=2e-5,
    num_train_epochs=1,
    max_steps=5000,
    load_best_model_at_end=True,
    eval_steps=100,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    use_cpu=False,
    label_smoothing_factor=0.0,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    
    dataloader_num_workers=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    optim="adamw_torch",
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={'use_reentrant':True}
    
)

student_trainer = DistillationTrainer(
    model=peft_model,
    args=student_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

student_trainer.train()

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss,Accuracy,F1 Score
100,2.350200,2.346023,0.225000,0.091954
200,2.363400,2.335537,0.225000,0.091837
300,2.334400,2.305827,0.229687,0.101449
400,2.121200,1.431415,0.815625,0.819634
500,0.801600,0.389147,0.901563,0.902739
600,0.421600,0.308108,0.903125,0.904414
700,0.345800,0.293711,0.909375,0.910286
800,0.322900,0.279425,0.910937,0.911901
900,0.344900,0.277893,0.909375,0.910491
1000,0.324500,0.265269,0.914062,0.915005


TrainOutput(global_step=5000, training_loss=0.4159150604248047, metrics={'train_runtime': 1116.3879, 'train_samples_per_second': 71.66, 'train_steps_per_second': 4.479, 'total_flos': 3959176047708672.0, 'train_loss': 0.4159150604248047, 'epoch': 0.6704210244033253})

In [19]:
# Setup Training args
#output_dir = "results"
#training_args = TrainingArguments(
#    output_dir=output_dir,
#    report_to="none",
#    eval_strategy='steps',
#    logging_steps=100,
#    learning_rate=2e-5,
#    num_train_epochs=1,
#    max_steps=1200,
#    use_cpu=False,
#    weight_decay=0.01,
#    dataloader_num_workers=4,
#    per_device_train_batch_size=16,
#    per_device_eval_batch_size=64,
 #   optim="adamw_torch",
#    gradient_checkpointing=False,
#    gradient_checkpointing_kwargs={'use_reentrant':True}
#)

#def get_trainer(model):
#      return  Trainer(
#          model=model,
#          args=training_args,
#          compute_metrics=compute_metrics,
#          train_dataset=train_dataset,
#          eval_dataset=eval_dataset,
#          data_collator=data_collator,
#      )

### Start Training

In [20]:
#peft_lora_finetuning_trainer = get_trainer(peft_model)

#result = peft_lora_finetuning_trainer.train()

## Evaluate Finetuned Model


### Performing Inference on Custom Input
Uncomment following functions for running inference on custom inputs

In [21]:
# def classify(model, tokenizer, text):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt").to(device)
#     output = model(**inputs)

#     prediction = output.logits.argmax(dim=-1).item()

#     print(f'\n Class: {prediction}, Label: {id2label[prediction]}, Text: {text}')
#     return id2label[prediction]

In [22]:
# classify( peft_model, tokenizer, "Kederis proclaims innocence Olympic champion Kostas Kederis today left hospital ahead of his date with IOC inquisitors claiming his ...")
# classify( peft_model, tokenizer, "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.")

### Run Inference on eval_dataset

In [23]:
from torch.utils.data import DataLoader
import evaluate
from tqdm import tqdm

def evaluate_model(inference_model, dataset, labelled=True, batch_size=8, data_collator=None):
    """
    Evaluate a PEFT model on a dataset.

    Args:
        inference_model: The model to evaluate.
        dataset: The dataset (Hugging Face Dataset) to run inference on.
        labelled (bool): If True, the dataset includes labels and metrics will be computed.
                         If False, only predictions will be returned.
        batch_size (int): Batch size for inference.
        data_collator: Function to collate batches. If None, the default collate_fn is used.

    Returns:
        If labelled is True, returns a tuple (metrics, predictions)
        If labelled is False, returns the predictions.
    """
    # Create the DataLoader
    eval_dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    inference_model.to(device)
    inference_model.eval()

    all_predictions = []
    if labelled:
        metric = evaluate.load('accuracy')

    # Loop over the DataLoader
    for batch in tqdm(eval_dataloader):
        # Move each tensor in the batch to the device
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = inference_model(**batch)
        predictions = outputs.logits.argmax(dim=-1)
        all_predictions.append(predictions.cpu())

        if labelled:
            # Expecting that labels are provided under the "labels" key.
            references = batch["labels"]
            metric.add_batch(
                predictions=predictions.cpu().numpy(),
                references=references.cpu().numpy()
            )

    # Concatenate predictions from all batches
    all_predictions = torch.cat(all_predictions, dim=0)

    if labelled:
        eval_metric = metric.compute()
        print("Evaluation Metric:", eval_metric)
        return eval_metric, all_predictions
    else:
        return all_predictions

In [24]:
# Check evaluation accuracy
_, _ = evaluate_model(peft_model, eval_dataset, True, 8, data_collator)

100%|██████████| 80/80 [00:02<00:00, 37.83it/s]


Evaluation Metric: {'accuracy': 0.9203125}


### Run Inference on unlabelled dataset

In [25]:
#Load your unlabelled data
unlabelled_dataset = pd.read_pickle("/kaggle/input/test-unlabelled/test_unlabelled.pkl")
test_dataset = unlabelled_dataset.map(preprocess, batched=False, remove_columns=["text"])
unlabelled_dataset

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 8000
})

In [26]:
# Run inference and save predictions
preds = evaluate_model(peft_model, test_dataset, False, 8, data_collator)
df_output = pd.DataFrame({
    'ID': range(len(preds)),
    'Label': preds.numpy()  # or preds.tolist()
})
df_output.to_csv(os.path.join("student_model","inference_output1.csv"), index=False)
print("Inference complete. Predictions saved to inference_output1.csv")

100%|██████████| 1000/1000 [00:34<00:00, 28.82it/s]

Inference complete. Predictions saved to inference_output1.csv
